In [330]:
!pip install sentence-transformers faiss-cpu networkx transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [331]:
import numpy as np
import pandas as pd
import faiss
import pickle
import networkx as nx
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline

In [332]:
final_df = pd.read_pickle("rag_data.pkl")

embeddings = np.load("embeddings.npy")

index = faiss.read_index("faiss_index.idx")

with open("kg_final.pkl", "rb") as f:
    G = pickle.load(f)

with open("centroids.pkl", "rb") as f:
    cluster_centroids = pickle.load(f)    
    



In [333]:
model = SentenceTransformer("all-MiniLM-L6-v2")

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model_llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10766.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 12796.06it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 9024.00it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in

In [334]:
cache = {}

In [335]:
def rewrite_query_llm(query):
    
    prompt = f"Rewrite this agricultural query in clear English: {query}"
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    
    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=30
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [336]:
def embed(query):
    return model.encode([query], normalize_embeddings=True)

In [337]:
def faiss_retrieve(query, k=5):
    
    q_emb = embed(query)
    
    D, I = index.search(q_emb, k)
    
    return final_df.iloc[I[0]]

In [338]:
def detect_language(query):
    # simple Hinglish/Hindi detection
    hindi_words = ["hai", "ka", "ki", "kya", "kaise", "ilaj", "kare", "mein"]

    if any(w in query.lower() for w in hindi_words):
        return "hinglish"
    
    return "english"

In [340]:
def kg_retrieve(query, cluster_centroids):

    import numpy as np

    q_emb = embed(query).flatten()   # ✅ fix shape

    best_cluster = None
    best_sim = -1

    for c, centroid in cluster_centroids.items():

        centroid = np.array(centroid).flatten()  # ✅ fix shape

        sim = np.dot(q_emb, centroid)

        # ❌ skip invalid
        if np.isnan(sim):
            continue

        if sim > best_sim:
            best_sim = sim
            best_cluster = c

    # ❌ no valid cluster found
    if best_cluster is None:
        return []

    node = f"cluster_{best_cluster}"

    # ❌ node not in graph
    if node not in G:
        return []

    answers = []

    for n in G.neighbors(node):
        if n in G.nodes and "type" in G.nodes[n] and G.nodes[n]["type"] == "answer":
            answers.append(n)

    return answers

In [343]:
# def merge_results(faiss_df, kg_results):
#     faiss_answers = faiss_df["clean_answer"].tolist()

#     combined = []
#     for ans in faiss_answers + kg_results:
#         if ans not in combined:
#             combined.append(ans)

#     return combined

In [ ]:
# def merge_results(faiss_df, kg_results):
#     faiss_answers = faiss_df["clean_answer"].tolist()

#     combined = []

#     for ans in faiss_answers + kg_results:
#         if isinstance(ans, str) and ans.strip() != "":
#             if ans not in combined:
#                 combined.append(ans)

#     return combined

In [372]:
def merge_results(faiss_df, kg_results):
    
    faiss_answers = faiss_df["clean_answer"].tolist()
    combined = faiss_answers[:3] + kg_results[:2]
    
    return list(set(combined))

In [424]:
def rerank(query, candidates, k=5):
    pairs = [(query, c) for c in candidates]
    scores = cross_encoder.predict(pairs)

    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [x[0] for x in ranked[:k]]

In [346]:
def build_context(results):
    return "\n".join(results)

In [403]:
# def rerank(query, candidates, k=5):

#     # ✅ ensure valid candidates
#     candidates = [c for c in candidates if isinstance(c, str) and c.strip() != ""]
#     if len(candidates) == 0:
#         return []

#     scored = []   # ✅ correctly defined

#     for ans in candidates:

#         # 🔹 base score from cross encoder
#         score = cross_encoder.predict([(query, ans)])[0]

#         # 🔥 boost if numbers match
#         q_nums = set([w for w in query.split() if w.isdigit()])
#         a_nums = set([w for w in ans.split() if w.isdigit()])

#         if len(q_nums & a_nums) > 0:
#             score += 0.1

#         scored.append((ans, score))   # ✅ inside loop

#     # 🔹 sort
#     ranked = sorted(scored, key=lambda x: x[1], reverse=True)

#     return [x[0] for x in ranked[:k]]

In [347]:
# def filter_answers(query, answers):
    
#     query_words = set(query.split())
    
#     filtered = []
    
#     for ans in answers:
#         if any(word in ans for word in query_words):
#             filtered.append(ans)
    
#     return filtered

In [348]:
def filter_answers(query, answers):

    keywords = set(query.lower().split())

    filtered = []

    for ans in answers:
        words = set(ans.lower().split())

        overlap = len(keywords & words)

        if overlap >= 2 and len(ans.split()) > 6:
            filtered.append(ans)

    return filtered

In [349]:
def clean_answers(answers):
    
    cleaned = []
    
    for ans in answers:
        ans = ans.replace("and", ",")
        ans = ans.strip()
        cleaned.append(ans)
    
    return cleaned

In [350]:
import re

def preprocess_context(answers):
    cleaned = []

    corrections = {
        "litar": "liter",
        "litre": "liter",
        "gm": "gram",
        "gms": "gram",
        "mlit": "ml",
        "spry": "spray",
        "spre": "spray",
        "watar": "water"
    }

    for ans in answers:
        ans = ans.lower()

        for w, r in corrections.items():
            ans = ans.replace(w, r)

        ans = re.sub(r"[^a-z0-9\s\.]", " ", ans)

        words = ans.split()
        # words = list(dict.fromkeys(words))  # remove duplicates

        cleaned.append(" ".join(words))

    return cleaned

In [353]:
def generate_answer(query, answers):

    # detect language (very simple)
    is_hinglish = any(word in query.lower() for word in ["hai", "ka", "kya", "mein", "ke"])

    answers = preprocess_context(answers)

    if len(answers) == 0:
        return "Not enough information. Please provide crop type, symptoms, or stage."

    # limit context
    answers = answers[:5]
    context = "\n".join(answers)

    if is_hinglish:
        instruction = "Answer in simple Hinglish."
    else:
        instruction = "Answer in clear English."

    prompt = f"""
You are an agriculture expert.

Convert the following raw points into a proper answer.

Rules:
- Combine similar points
- Fix grammar and spelling
- Do NOT repeat
- Keep it practical
- Keep chemical names intact
- Use bullet points

{instruction}

Context:
{context}

Final Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    inputs = {k: v.to(model_llm.device) for k, v in inputs.items()}
    outputs = model_llm.generate(
    **inputs,
    max_new_tokens=120,
    min_new_tokens=60,        # ✅ ensures not too short
    do_sample=False,          # ✅ removes randomness
    num_beams=4,              # ✅ better structured output
    early_stopping=True
)


    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Final Answer:" in result:
        result = result.split("Final Answer:")[-1].strip()

    return result

In [404]:
def run_pipeline(query, cluster_centroids, k=5):

    rewritten = rewrite_query_llm(query)

    # 🔥 try multiple k (boost retrieval quality)
    faiss_res = faiss_retrieve(rewritten, k=7)

    kg_res = kg_retrieve(rewritten, cluster_centroids)

    combined = merge_results(faiss_res, kg_res)

    final_answers = rerank(rewritten, combined, k=5)

    answer = generate_answer(query, final_answers)

    return {
        "original_query": query,
        "rewritten_query": rewritten,
        "answers": final_answers,
        "final_answer": answer
    }

In [405]:
output = run_pipeline("yellowing of leaf in paddy ", cluster_centroids)



In [406]:
print("\n🔹 Rewritten Query:")
print(output["rewritten_query"])

print("\n🔹 Retrieved Answers:")
for i, ans in enumerate(output["answers"], 1):
    print(f"{i}. {ans}")

print("\n🔹 Final Answer:")
print(output["final_answer"])


🔹 Rewritten Query:
Yellowing of leaf in paddy

🔹 Retrieved Answers:
1. use zinksulphate and urea
2. spray 2 kilogram zinc sulphate 8 kilogram ureaacre in 400 liter water
3. carbendazim 2 mllit along with 10 gram urealit for spray
4. carbendazim 50 wp 2 gmlitwaterchlorpyriphos 20 ec 4 mllitwater
5. 1 carbendajim 12 maincojeb 63 wp 2 gram litar water 2 streptocyclin 1 gram prati 10 litar water me gholkar spre kare

🔹 Final Answer:
Use zinc sulphate and urea spray 2 kilogram zinc sulphate 8 kilogram ureaacre in 400 liter water carbendazim 2 mllit along with 10 gram urealit for spray carbendazim 50 wp 2 gramlwaterchlorpyriphos 20 ec 4 mllitwater 1 carbendajim 12 maincojeb 63 wp 2 gram liter water 2 streptocyclin 1 gram prati 10 liter water me g


In [377]:
output = run_pipeline(" rice mai stem borer ka ilaj", cluster_centroids)

In [378]:
print("\n🔹 Rewritten Query:")
print(output["rewritten_query"])

print("\n🔹 Retrieved Answers:")
for i, ans in enumerate(output["answers"], 1):
    print(f"{i}. {ans}")

print("\n🔹 Final Answer:")
print(output["final_answer"])


🔹 Rewritten Query:
rice mai stem borer ka ilaj

🔹 Retrieved Answers:
1. kisan bhai aap rice me cartap hydrochloride 50 sp 400 mlacre ki dar se spray kare
2. lamda caylohetrin 2 mililitre liter water ke sath milaakar chidcav kare
3. trichocard lga sakte ho
4. suggested to spray hitcel 1 mllitre of water
5. advised to apply chloropyriphos 20 ec 2 milliliter per liter of water or monocrotophos 40 ec 3 milliliter per liter of water

🔹 Final Answer:
kisan bhai aap rice me cartap hydrochloride 50 sp 400 mlacre ki dar se spray kare lamda caylohetrin 2 mililiter liter water ke sath milaakar chidcav kare trichocard lga sakte ho suggested to spray hitcel 1 mlliter of water advised to apply chloropyriphos 20 ec 2 milliliter per liter of water or


In [379]:
output = run_pipeline(" there is excess water in field", cluster_centroids)

In [380]:
print("\n🔹 Rewritten Query:")
print(output["rewritten_query"])

print("\n🔹 Retrieved Answers:")
for i, ans in enumerate(output["answers"], 1):
    print(f"{i}. {ans}")

print("\n🔹 Final Answer:")
print(output["final_answer"])


🔹 Rewritten Query:
There is excess water in the field.

🔹 Retrieved Answers:
1. drain out excess water from field
2. drain water from soyabean field
3. recommended eradicate water from field opt for help from aao
4. muddy water in rice field for more details contact with nearest assistant agriculture officer
5. advised to spray ekalux 2 milliliter liter of water

🔹 Final Answer:
Drain out excess water from field drain water from soyabean field recommended eradicate water from field opt for help from aao muddy water in rice field for more details contact with nearest assistant agriculture officer advised to spray ekalux 2 milliliter liter of water final.


In [381]:
import pandas as pd

test_df = pd.read_csv("test_data.csv")

In [382]:
print(test_df.columns)
test_df.head()

Index(['state', 'query', 'answer', 'clean_query', 'clean_answer'], dtype='object')


,state,query,answer,clean_query,clean_answer
0,ANDHRA PRADESH,asked about nursery management in rice,information provided as per data,nursery management in rice,NaN
1,ASSAM,STEM BORER PROBLEM IN RICE,SPRAY MONOCROTOPHOS 20 EC2MLL IN RICE,stem borer problem in rice,spray monocrotophos 20 ec 2 mll in rice
2,ASSAM,ASKING ABOUT THE STEM BORER PROBLEM IN RICE,APPLICATION OF CHLOROPYRIPHOS 20 EC 4 ML PER ...,asking about the stem borer problem in rice,application of chloropyriphos 20 ec 4 millilit...
3,ASSAM,Asking about the control merasure of stem borrer,Suggested to spray Rocket 2mllit of water,asking about the control merasure of stem borrer,suggested to spray rocket 2 mllit of water
4,ASSAM,PROBLEM OF GANDHI BUG IN RICE,ADVISED HIM TO APPLY JACK POT 2 GM LITWER OF...,problem of gandhi bug in rice,advised him to apply jack pot 2 gram litwer of...


In [383]:
test_df_eval = test_df[["clean_query", "clean_answer"]].copy()

test_df_eval.rename(columns={
    "clean_query": "query",
    "clean_answer": "ground_truth"
}, inplace=True)

In [384]:
test_df_eval = test_df_eval.dropna()

In [385]:
test_df_eval = test_df_eval.reset_index(drop=True)

In [386]:
print(test_df_eval.columns)
test_df_eval.head()
print("Size:", len(test_df_eval))

Index(['query', 'ground_truth'], dtype='object')
Size: 91310


In [387]:
test_df_eval.head()

,query,ground_truth
0,stem borer problem in rice,spray monocrotophos 20 ec 2 mll in rice
1,asking about the stem borer problem in rice,application of chloropyriphos 20 ec 4 millilit...
2,asking about the control merasure of stem borrer,suggested to spray rocket 2 mllit of water
3,problem of gandhi bug in rice,advised him to apply jack pot 2 gram litwer of...
4,asking about the control of stem borer,suggested to apply the ekalux 3 milliliter per...


In [388]:
# def is_match(gt, pred):

#     gt = gt.lower().strip()
#     pred = pred.lower().strip()

#     # strict: full containment OR strong overlap
#     if gt in pred:
#         return True

#     gt_words = set(gt.split())
#     pred_words = set(pred.split())

#     overlap = len(gt_words & pred_words)

#     # require strong match
#     return overlap >= max(3, len(gt_words) // 2)


# def hit_at_k(retrieved, gt, k=5):
#     for r in retrieved[:k]:
#         if is_match(gt, r):
#             return 1
#     return 0


# def reciprocal_rank(retrieved, gt):
#     for i, r in enumerate(retrieved):
#         if is_match(gt, r):
#             return 1 / (i + 1)
#     return 0

In [475]:
from sentence_transformers import SentenceTransformer, util

eval_model = SentenceTransformer("intfloat/multilingual-e5-base")

def hit_at_k_semantic(retrieved, ground_truth, k):
    gt = ground_truth.lower().strip()
    gt_words = set(gt.split())

    gt_emb = eval_model.encode(gt, convert_to_tensor=True)

    for ans in retrieved[:k]:
        ans = ans.lower().strip()
        ans_words = set(ans.split())

        # 1. semantic similarity
        ans_emb = eval_model.encode(ans, convert_to_tensor=True)
        score = util.cos_sim(gt_emb, ans_emb).item()

        # 2. keyword overlap
        overlap = len(gt_words & ans_words)

        # 3. number match (VERY IMPORTANT for your data)
        numbers_gt = set([w for w in gt.split() if w.isdigit()])
        numbers_ans = set([w for w in ans.split() if w.isdigit()])
        number_match = len(numbers_gt & numbers_ans)

        # if (
        #     score >= 0.85 and
        #     overlap >= 3 and
        #     (len(numbers_gt) == 0 or number_match >= 1)
        # ):
        # 🔥 balanced condition
        if (
            (score >= 0.85 and overlap >= 3 and (len(numbers_gt) == 0 or number_match >= 1)) or
            (score >= 0.70 and overlap >= 4)
        ):
      
            
            return 1

    return 0

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11211.55it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [476]:
# def hit_at_k_semantic(retrieved, ground_truth, k):

#     gt = ground_truth.lower().strip()
#     gt_emb = eval_model.encode(gt, convert_to_tensor=True)

#     for ans in retrieved[:k]:
#         ans = ans.lower().strip()
#         ans_emb = eval_model.encode(ans, convert_to_tensor=True)

#         score = util.cos_sim(gt_emb, ans_emb).item()

#         # 🔥 relaxed but still meaningful
#         if score >= 0.70:
#             return 1

#     return 0

In [483]:
def reciprocal_rank_semantic(retrieved, ground_truth):
    gt = ground_truth.lower().strip()
    gt_words = set(gt.split())

    gt_emb = eval_model.encode(gt, convert_to_tensor=True)

    for i, ans in enumerate(retrieved):
        ans = ans.lower().strip()
        ans_words = set(ans.split())

        ans_emb = eval_model.encode(ans, convert_to_tensor=True)
        score = util.cos_sim(gt_emb, ans_emb).item()

        overlap = len(gt_words & ans_words)

        numbers_gt = set([w for w in gt.split() if w.isdigit()])
        numbers_ans = set([w for w in ans.split() if w.isdigit()])
        number_match = len(numbers_gt & numbers_ans)

        # if (
        #     score >= 0.85 and
        #     overlap >= 3 and
        #     (len(numbers_gt) == 0 or number_match >= 1)
        # ):
          if (
            (score >= 0.85 and overlap >= 3 and (len(numbers_gt) == 0 or number_match >= 1)) or
            (score >= 0.70 and overlap >= 4)
        ):
            
            return 1 / (i + 1)

    return 0

IndentationError: unexpected indent (3418766072.py, line 25)

In [484]:
# def run_pipeline_fast(query, cluster_centroids, k=5):

#     # ❌ no rewrite (slow)
#     # ❌ no cross encoder (slow)

#     q = query.lower()

#     faiss_res = faiss_retrieve(q, k)
#     kg_res = kg_retrieve(q, cluster_centroids)

#     combined = list(set(faiss_res["clean_answer"].tolist() + kg_res))

#     return combined[:k]

In [485]:
def run_pipeline_fast(query, cluster_centroids, k=5):

    q = query.lower()

    faiss_res = faiss_retrieve(q, 10)   # higher recall
    kg_res = kg_retrieve(q, cluster_centroids)

    combined = merge_results(faiss_res, kg_res)

    final = rerank(q, combined, k)

    return final

In [486]:
sample_test = test_df_eval.sample(500, random_state=42)

In [487]:
def evaluate(sample_df, cluster_centroids, k=5):

    hit_scores = []
    mrr_scores = []

    for _, row in sample_df.iterrows():

        query = row["query"]
        gt = row["ground_truth"]

        retrieved = run_pipeline_fast(query, cluster_centroids, k)

        hit_scores.append(hit_at_k_semantic(retrieved, gt, k))
        mrr_scores.append(reciprocal_rank_semantic(retrieved, gt))

    hit_k = sum(hit_scores) / len(hit_scores)
    mrr = sum(mrr_scores) / len(mrr_scores)

    print(f"Hit@{k}:", round(hit_k, 4))
    print("MRR:", round(mrr, 4))

In [488]:
evaluate(sample_test, cluster_centroids, k=5)

Hit@5: 0.824
MRR: 0.7472


In [489]:
import pandas as pd

train_df = pd.read_csv("train_data.csv")

In [490]:
overlap = set(test_df_eval["query"]).intersection(set(train_df["clean_query"]))
print("Overlap:", len(overlap))

Overlap: 0


In [491]:
for i in range(5):
    row = sample_test.iloc[i]
    
    query = row["query"]
    gt = row["ground_truth"]
    
    retrieved = run_pipeline_fast(query, cluster_centroids, k=5)
    
    print("\nQUERY:", query)
    print("GT:", gt)
    print("TOP RESULT:", retrieved[0])


QUERY: give me weather information
GT: no rain possibility in next 5 days
TOP RESULT: no rain possibility in next 5 days

QUERY: leaf folder management in rice
GT: recommended to spray indoxacarb avaunt 200 milliliter 200 litres of water acre 200 200
TOP RESULT: spray cholorophiryphas 40 mlpump

QUERY: control of bph in rice
GT: recommended for spray acephate 5 imidacloprid 11 61 ec 25 milliliter lit of water
TOP RESULT: transfer to agriculture expert

QUERY: bacterial disease of rice
GT: recommended for spray trichoderma viride 5 gmlit of water and pseudomonas fluorescence 5 gmlit of water
TOP RESULT: recommended to spray 30 grams copper oxy chloride 1 gram streptocyclin per 10 litres

QUERY: information regarding for the seed treatment of paddyjhona
GT: soak the selected seed in 10 liter of water contianing 20 gram bavistin 50 wp 1 gram streptocycline for 8 to 10 hours before sowing
TOP RESULT: 8 kilogram seeds should be soaked in 10 litres of water containing 20 gram bavistin and 1

In [492]:
query = sample_test.iloc[0]["query"]
gt = sample_test.iloc[0]["ground_truth"]
preds = run_pipeline_fast(query, cluster_centroids, k=3)

print("GT:", gt)
print("PRED:", preds)

GT: no rain possibility in next 5 days
PRED: ['no rain possibility in next 5 days', '10 chance of rainfall today', 'this week is unlikely to rain']


In [496]:
query = sample_test.iloc[7]["query"]

q = query.lower()
faiss_res = faiss_retrieve(q, 30)
kg_res = kg_retrieve(q, cluster_centroids)

combined = merge_results(faiss_res, kg_res)

print("BEFORE RERANK:")
for i, c in enumerate(combined[:10]):
    print(i, c)

print("\nAFTER RERANK:")
reranked = rerank(q, combined, k=10)
for i, c in enumerate(reranked):
    print(i, c)

BEFORE RERANK:
0 sapray 1 liter chloropyriphose in 100 liter of wateracre
1 advised to spray luphos 25 milliliter liter of water
2 leaf yellow can be minimized by spraying 2 urea mixed with mancozeb at 25 gmlit
3 advised to spray tricel 20 ec 2 milliliter liter of water
4 zinc sulphate 200 gramacre 180 li water me spray kare

AFTER RERANK:
0 leaf yellow can be minimized by spraying 2 urea mixed with mancozeb at 25 gmlit
1 zinc sulphate 200 gramacre 180 li water me spray kare
2 advised to spray luphos 25 milliliter liter of water
3 advised to spray tricel 20 ec 2 milliliter liter of water
4 sapray 1 liter chloropyriphose in 100 liter of wateracre
